# Train Giano and use the resulting weights
This notebook takes one optimiser step for each official ensemble member on eight real Trentino training windows and evaluates it on eight separate 2011 development windows. That is enough to trace the path from measurements to a usable model. The published results use thousands of windows and many rounds.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from giano.data import load_input
from giano.fill import fill
from giano.train import train
from giano.trentino import PROTOCOL

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
workspace = TemporaryDirectory()
run = Path(workspace.name)
cache = run / "windows"
for split in ("train", "val"):
    (cache / split).mkdir(parents=True)
with np.load(root / "examples/trentino/training-preview.npz") as sample:
    for key in sample.files:
        split, field = key.split("__")
        np.save(cache / split / f"{field}.npy", sample[key])
(cache / "recipe.json").write_text(json.dumps({"protocol": 1, "preview": True}))

## What is being learned?
Each window has 72 hourly steps, 24 station slots and six meteorological variables. Read the quality mask to count the measured targets available for supervision. The loss uses the first four variables. Wind contributes inputs and uses the physical reference at inference.

In [ ]:
variables = [
    "temperature",
    "humidity",
    "pressure",
    "precipitation",
    "wind_speed",
    "wind_direction",
]
pd.DataFrame(
    {
        split: np.load(cache / split / "target_mask.npy").sum(axis=(0, 1))
        for split in ("train", "val")
    },
    index=variables,
)

## Fit and export
Training builds the same architecture as the released predictor. For this preview, each seed sees one batch of eight windows.

In [ ]:
trained = run / "training"
train(cache, trained)
report = json.loads((trained / "training.json").read_text())
pd.DataFrame(
    [
        dict(seed=seed, **result["history"][0])
        for seed, result in report["seeds"].items()
    ]
)

## Use the weights you just produced
Load the Swiss observations and ask the standard inference function to fill natural temperature gaps in March using your ensemble.

In [ ]:
swiss = root / "examples/noaa-2024"
ds = load_input(swiss / "observations.csv", swiss / "stations.csv")
out = fill(
    ds, "2024-03-01", "2024-04-01", ["temperature"], ensemble=trained / "ensemble.pt"
)
print(
    "Used freshly trained weights:",
    out.attrs["ensemble_sha256"] == report["ensemble_sha256"],
)
pd.Series(out.temperature_method.values.ravel()).value_counts()

## Inspect the full development run
The following curves are the archived development scores of the official 36-round runs. Each point uses the four supervised variables on two 2011 cohorts. Mark the lowest score for each seed. These are the stopping rounds used for the released ensemble.

In [ ]:
curves = json.loads((PROTOCOL / "learning-curves.json").read_text())
fig, ax = plt.subplots()
for seed, curve in curves.items():
    scores = curve["development_mae"]
    (line,) = ax.plot(range(1, len(scores) + 1), scores, label=f"Seed {seed}")
    selected = curve["selected_round"]
    ax.scatter(selected, scores[selected - 1], color=line.get_color())
ax.set(xlabel="Training round", ylabel="Normalised development MAE")
ax.legend()
plt.show()
workspace.cleanup()

## Repeat with the full dataset
Download the Meteotrentino folders from Drive. Run the following commands from the repository root. To repeat selection on development data, omit the fixed-rounds option. Notebook 03 explains how to inspect a reconstruction. Notebook 04 measures error on hidden observations. [Dataset on Google Drive](https://drive.google.com/drive/folders/1DPsYf6dVRtCqxE3mDYAx4evTj0WASXK7?usp=sharing) · [Training guide](https://github.com/itsminni/giano/wiki/Training-and-reproduction)

```sh
uv run --locked giano-prepare --data data --output runs/windows
uv run --locked giano-train --cache runs/windows --output runs/training --fixed-rounds
uv run --locked giano-evaluate --data data --ensemble runs/training/ensemble.pt --local-context --station-reference --output runs/evaluation
```